In [ ]:
from TTS.api import TTS
import torch, re

# ────────────────────────────────
# ⚙️  Настройки
MODEL = "tts_models/multilingual/multi-dataset/xtts_v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SRC_TXT = r"input.txt"

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    text = re.sub(r"\s+", " ", text).strip()
    return text


TEXT = "Это пример текста для синтеза речи. Измените его на свой образец."

LANG = "ru"
SPEAKER = "Adde Michal"     # имя любого доступного спикера
SPEAKER_WAV = None          # путь к своему голосу (если клонируешь)
STYLE_WAV = None            # путь к аудио с нужной интонацией
OUTPUT_FILE = "xtts_full_test.wav"

# 🔧 Только поддерживаемые XTTS параметры
# (остальные игнорируются моделью)
TEMPERATURE = 1.1          # «температура» — вариативность голоса (0=монотонно, 1=естественно)
TOP_P = 0.8                # sampling nucleus — контролирует «естественность» речи
TOP_K = 50                  # ограничение выборки слов
REPETITION_PENALTY = 1.0    # штраф за повтор слов
LENGTH_PENALTY = 1.5        # как сильно модель "боится" длинных предложений
SPEED = 1.0                 # скорость речи (если активна)
ENABLE_TEXT_SPLITTING = False # автоматически делит длинный текст на фразы


# ────────────────────────────────
# 🚀 Инициализация модели
tts = TTS(MODEL).to(DEVICE)

# 🎙️ Генерация
tts.tts_to_file(
    text=TEXT,
    speaker=SPEAKER,
    language=LANG,
    file_path=OUTPUT_FILE,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    repetition_penalty=REPETITION_PENALTY,
    # length_penalty=LENGTH_PENALTY,
    speed=SPEED,
    enable_text_splitting=ENABLE_TEXT_SPLITTING,
    split_sentences = False
)

print(f"Сохранено: {OUTPUT_FILE}")

In [ ]:
WINDOW = 180
words = TEXT.split()
rolling_chunks = []
for i in range(len(words)):
    cur = words[i]
    j = i + 1
    while j < len(words) and len(cur) + 1 + len(words[j]) <= WINDOW:
        cur += " " + words[j]
        j += 1
    rolling_chunks.append(cur)

print(f"created {len(rolling_chunks)} rolling chunks")

In [ ]:
import random, os
SEED = 42  # поменяйте при необходимости

# фиксируем сиды
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# сделать поведение cudnn более детерминированным
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

WINDOW = 180
words = TEXT.split()
rolling_chunks = []
for i in range(len(words)):
    cur = words[i]
    j = i + 1
    while j < len(words) and len(cur) + 1 + len(words[j]) <= WINDOW:
        cur += " " + words[j]; j += 1
    rolling_chunks.append(cur)

out_dir = r"rolling_audio"
os.makedirs(out_dir, exist_ok=True)

for i, chunk in enumerate(rolling_chunks[:193], 1):
    path = os.path.join(out_dir, f"roll_{i:03}.wav")
    tts.tts_to_file(text=chunk, speaker=SPEAKER, language=LANG, file_path=path,
                    temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                    repetition_penalty=REPETITION_PENALTY, speed=SPEED,
                    enable_text_splitting=False, split_sentences=False)
    print(f"{i}/{min(193,len(rolling_chunks))} saved", end="\r")

In [ ]:
# ...existing code...
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa

WAV_DIR = "rolling_audio"  # поменяйте при необходимости

files = sorted([os.path.join(WAV_DIR, f) for f in os.listdir(WAV_DIR) if f.lower().endswith(".wav")])
if len(files) < 1:
    print("Нет .wav файлов в папке:", WAV_DIR)
else:
    # взять первые два файла (если второго нет — покажет только первый)
    to_plot = files[:2]
    plt.figure(figsize=(12, 5 * len(to_plot)))
    for i, path in enumerate(to_plot, start=1):
        y, sr = librosa.load(path, sr=None, mono=True)
        t = np.arange(len(y)) / sr
        ax = plt.subplot(len(to_plot), 1, i)
        ax.plot(t, y, linewidth=0.1)
        ax.set_title(f"Waveform {i}: {os.path.basename(path)} (sr={sr})")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")
        ax.grid(True)
    plt.tight_layout()
    plt.show()
# ...existing code...

In [ ]:
# ...existing code...
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy import signal

WAV_DIR = "rolling_audio"  # поменяйте при необходимости

def rms_envelope(path, frame_ms=8, hop_ms=2):
    y, sr = librosa.load(path, sr=None, mono=True)
    frame_length = max(1, int(sr * frame_ms / 1000))
    hop_length = max(1, int(sr * hop_ms / 1000))
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)
    return rms, times, sr, hop_length, y

def norm(x):
    x = np.array(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

files = sorted([os.path.join(WAV_DIR, f) for f in os.listdir(WAV_DIR) if f.lower().endswith(".wav")])
if len(files) < 2:
    print("Нужно минимум 2 .wav файла в", WAV_DIR)
else:
    a_rms, a_t, sr, hop, a_y = rms_envelope(files[0])
    b_rms, b_t, sr2, hop2, b_y = rms_envelope(files[1])
    assert sr == sr2 and hop == hop2

    # корреляция огибающих (RMS)
    corr = np.correlate(norm(a_rms), norm(b_rms), mode="full")
    lag_idx = corr.argmax() - (len(b_rms) - 1)
    lag_seconds_env = lag_idx * hop / sr

    # грубая корреляция сигналов (с декимацией для скорости)
    decim_factor = max(1, sr // 8000)
    a_dec = signal.decimate(a_y, decim_factor, ftype='iir', zero_phase=True) if decim_factor > 1 else a_y
    b_dec = signal.decimate(b_y, decim_factor, ftype='iir', zero_phase=True) if decim_factor > 1 else b_y
    corr_sig = signal.correlate(norm(a_dec), norm(b_dec), mode='full')
    lag_idx_sig = corr_sig.argmax() - (len(b_dec) - 1)
    lag_seconds_sig = lag_idx_sig / (sr / decim_factor)

    print(f"Offset by RMS-envelope: {lag_idx} frames  => {lag_seconds_env:.3f} s")
    print(f"Offset by raw-signal (decim x{decim_factor}): {lag_idx_sig} samples_decim => {lag_seconds_sig:.3f} s")

    # Показать огибающие и наложение с найденным сдвигом (RMS)
    plt.figure(figsize=(12,5))
    plt.subplot(2,1,1)
    plt.plot(a_t, 20*np.log10(a_rms+1e-9), label="file 1 (dB)")
    plt.plot(b_t, 20*np.log10(b_rms+1e-9), label="file 2 (dB)")
    plt.title("RMS envelopes (raw)")
    plt.legend(); plt.grid(True)

    plt.subplot(2,1,2)
    # сдвинем вторую огибающую по найденному лагу (RMS estimate)
    b_t_shifted = b_t + lag_seconds_env
    plt.plot(a_t, norm(20*np.log10(a_rms+1e-9)), label="file 1 (norm)")
    plt.plot(b_t_shifted, norm(20*np.log10(b_rms+1e-9)), label=f"file 2 shifted by {lag_seconds_env:.3f}s (norm)", alpha=0.8)
    plt.title("Normalized envelopes (overlap with RMS-estimated shift)")
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.show()
# ...existing code...

In [ ]:
# ...existing code...
plt.figure(figsize=(12,5))
plt.subplot(2,1,1)
plt.plot(a_t, 20*np.log10(a_rms+1e-9), label="file 1 (dB)")
plt.plot(b_t, 20*np.log10(b_rms+1e-9), label="file 2 (dB)")
plt.title("RMS envelopes (raw)")
plt.xlim(0, 1)  # показывать только первую секунду
plt.legend(); plt.grid(True)

plt.subplot(2,1,2)
b_t_shifted = b_t + lag_seconds_env
plt.plot(a_t, norm(20*np.log10(a_rms+1e-9)), label="file 1 (norm)")
plt.plot(b_t_shifted, norm(20*np.log10(b_rms+1e-9)), label=f"file 2 shifted by {lag_seconds_env:.3f}s (norm)", alpha=0.8)
plt.title("Normalized envelopes (overlap with RMS-estimated shift)")
plt.xlim(0, 1)  # показывать только первую секунду
plt.legend(); plt.grid(True)

# Добавим более частую сетку для детализации
plt.grid(True, which='major', linestyle='-', alpha=0.8)
plt.grid(True, which='minor', linestyle=':', alpha=0.5)
plt.minorticks_on()

plt.tight_layout()
plt.show()
# ...existing code...

In [ ]:
from IPython.display import Audio
y, sr = librosa.load(files[0], sr=None)
offset_samples = int(lag_seconds_env * sr)
trimmed = y[offset_samples:] if offset_samples >= 0 else np.pad(y, (abs(offset_samples), 0))
Audio(trimmed, rate=sr)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa

WAV_DIR = "rolling_audio"

files = sorted([os.path.join(WAV_DIR, f) for f in os.listdir(WAV_DIR) if f.lower().endswith(".wav")])
if len(files) < 1:
    print("Нет .wav файлов в папке:", WAV_DIR)
else:
    to_plot = files[:2]
    plt.figure(figsize=(12, 8 * len(to_plot)))
    
    for i, path in enumerate(to_plot, start=1):
        y, sr = librosa.load(path, sr=None, mono=True)
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        
        ax = plt.subplot(len(to_plot), 1, i)
        librosa.display.specshow(D, sr=sr, y_axis='log', x_axis='time', ax=ax)
        ax.set_title(f"Spectrogram {i}: {os.path.basename(path)}")
        plt.colorbar(ax.collections[0], ax=ax, format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import librosa
from scipy import signal

def find_overlap(file1, file2, window_ms=50):
    # Загрузка аудио
    y1, sr = librosa.load(file1, sr=None)
    y2, _ = librosa.load(file2, sr=None)
    
    # Параметры для RMS огибающей
    frame_length = int(sr * window_ms / 1000)  # окно для RMS (в сэмплах)
    hop_length = frame_length // 4  # перекрытие 75%
    
    # Получение RMS огибающих
    rms1 = librosa.feature.rms(y=y1, frame_length=frame_length, hop_length=hop_length)[0]
    rms2 = librosa.feature.rms(y=y2, frame_length=frame_length, hop_length=hop_length)[0]
    
    # Нормализация для корреляции
    rms1_norm = (rms1 - rms1.mean()) / rms1.std()
    rms2_norm = (rms2 - rms2.mean()) / rms2.std()
    
    # Кросс-корреляция
    correlation = signal.correlate(rms1_norm, rms2_norm, mode='full')
    lag_frames = correlation.argmax() - (len(rms2_norm) - 1)
    
    # Конвертация в миллисекунды
    lag_ms = (lag_frames * hop_length * 1000) / sr
    
    # Проверка точности через корреляцию после выравнивания
    if lag_frames >= 0:
        overlap1 = rms1_norm[lag_frames:lag_frames+len(rms2_norm)]
        overlap2 = rms2_norm
    else:
        overlap1 = rms1_norm[:len(rms2_norm)+lag_frames]
        overlap2 = rms2_norm[-lag_frames:]
    
    overlap_len = min(len(overlap1), len(overlap2))
    accuracy = np.corrcoef(overlap1[:overlap_len], overlap2[:overlap_len])[0,1]
    
    return lag_ms, accuracy

# Проверка для первых двух файлов
files = sorted([f for f in os.listdir(WAV_DIR) if f.endswith('.wav')])
if len(files) >= 2:
    file1, file2 = [os.path.join(WAV_DIR, f) for f in files[:2]]
    lag_ms, accuracy = find_overlap(file1, file2)
    print(f"Второй файл начинается с отметки {lag_ms:.1f} мс первого файла")
    print(f"Точность совпадения: {accuracy:.3f}")